# 멤버 주간 소비 지표 분석

이 노트북은 주간(7일) 소비 데이터를 분석하여 반복 소비 패턴, 요일별 추이, 낭비성 소비, 절약 가능성을 포함한 종합 지표를 JSON으로 출력합니다.

`data_index_day_json.ipynb`의 일간 이상 지출 탐지 결과를 주간 단위로 누적집계합니다.

In [1]:
from __future__ import annotations

import json
import pandas as pd
from pathlib import Path
from datetime import timedelta

# ===== 분석 파라미터 =====
MEMBER_ID: int = 1
WEEK_START: str = "2024-04-01"  # 분석할 주 시작일
WEEK_END: str   = "2024-04-07"  # 분석할 주 종료일

# ===== 가맹점 분류 키워드 =====
DELIVERY_KEYWORDS    = ['배달의민족', '쿠팡이츠']
CAFE_KEYWORDS        = ['스타벅스', '이디야', '투썸플레이스', '할리스', '커피빈', '폴바셋', '빽다방', '메가커피', '컴포즈']
CONVENIENCE_KEYWORDS = ['CU', 'GS25', '세븐일레븐', '미니스톱', 'emart24', '이마트24']
TAXI_KEYWORDS        = ['카카오택시', '타다', '우티']

MICRO_THRESHOLD: int = 10_000  # 소액 기준 (원)
LATE_NIGHT_HOUR: int = 21      # 야간 소비 시작 시간
WEEKDAY_NAMES = ['월', '화', '수', '목', '금', '토', '일']

print('✅ 설정 완료')

✅ 설정 완료


In [2]:
# ===== 데이터 로드 및 주차 필터링 =====
df_past  = pd.read_csv(r"C:\Users\user\dev\catcher-llm\data\raw\csv\consumption_v1.csv")
df_month = pd.read_csv(r"C:\Users\user\dev\catcher-llm\notebook\team02\data_pre\data_input_month.csv")

df_past_m  = df_past[df_past['멤버 id'] == MEMBER_ID].copy()
df_month_m = df_month[df_month['멤버 id'] == MEMBER_ID].copy()

df_all = pd.concat([df_past_m, df_month_m], ignore_index=True)
df_all['사용 시간'] = pd.to_datetime(df_all['사용 시간'])
df_all['사용 금액'] = pd.to_numeric(df_all['사용 금액'])
df_all['date']    = df_all['사용 시간'].dt.date
df_all['hour']    = df_all['사용 시간'].dt.hour
df_all['weekday'] = df_all['사용 시간'].dt.dayofweek  # 0=월, 6=일

week_start      = pd.to_datetime(WEEK_START).date()
week_end        = pd.to_datetime(WEEK_END).date()
prev_week_start = week_start - timedelta(days=7)
prev_week_end   = week_end   - timedelta(days=7)

df_this      = df_all[(df_all['date'] >= week_start)      & (df_all['date'] <= week_end)].copy()
df_prev      = df_all[(df_all['date'] >= prev_week_start) & (df_all['date'] <= prev_week_end)].copy()
df_past_only = df_all[df_all['date'] < week_start].copy()

# IQR 상한선 (이전 전체 데이터 기반 - 일간 노트북과 동일 방식)
q1  = float(df_past_only['사용 금액'].quantile(0.25))
q3  = float(df_past_only['사용 금액'].quantile(0.75))
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print(f"이번 주 ({WEEK_START} ~ {WEEK_END}): {len(df_this)}건 / 총 {df_this['사용 금액'].sum():,.0f}원")
print(f"전주   ({prev_week_start} ~ {prev_week_end}): {len(df_prev)}건 / 총 {df_prev['사용 금액'].sum():,.0f}원")
print(f"IQR 상한선 (고액 기준): {upper_bound:,.0f}원")

이번 주 (2024-04-01 ~ 2024-04-07): 42건 / 총 561,019원
전주   (2024-03-25 ~ 2024-03-31): 36건 / 총 352,341원
IQR 상한선 (고액 기준): 22,754원


## [1] 주간 총 지출 요약
전주 대비 총액, 일평균, 최고/최저 지출일을 파악합니다.

In [3]:
this_total  = float(df_this['사용 금액'].sum())
prev_total  = float(df_prev['사용 금액'].sum())
amount_diff = this_total - prev_total
diff_rate   = (amount_diff / prev_total * 100) if prev_total != 0 else 0.0

active_days = df_this['date'].nunique()
daily_avg   = this_total / active_days if active_days > 0 else 0.0

daily_sums = df_this.groupby('date')['사용 금액'].sum()
max_day_date   = str(daily_sums.idxmax()) if not daily_sums.empty else None
max_day_amount = int(daily_sums.max())    if not daily_sums.empty else 0
min_day_date   = str(daily_sums.idxmin()) if not daily_sums.empty else None
min_day_amount = int(daily_sums.min())    if not daily_sums.empty else 0

print("[주간 요약]")
print(f"  이번 주 총액  : {this_total:>12,.0f}원")
print(f"  전주 총액     : {prev_total:>12,.0f}원")
print(f"  증감          : {amount_diff:>+12,.0f}원  ({diff_rate:+.1f}%)")
print(f"  일평균        : {daily_avg:>12,.0f}원")
print(f"  최고 지출일   : {max_day_date}  ({max_day_amount:,.0f}원)")
print(f"  최저 지출일   : {min_day_date}  ({min_day_amount:,.0f}원)")
print(f"  결제 건수     : {len(df_this)}건")

[주간 요약]
  이번 주 총액  :      561,019원
  전주 총액     :      352,341원
  증감          :     +208,678원  (+59.2%)
  일평균        :       80,146원
  최고 지출일   : 2024-04-01  (133,044원)
  최저 지출일   : 2024-04-04  (53,026원)
  결제 건수     : 42건


## [2] 카테고리별 주간 분석
각 카테고리의 주간 지출액, 비중, 전주 대비 증감을 비교합니다.

In [4]:
this_cat     = df_this.groupby('업종 카테고리')['사용 금액'].sum()
prev_cat     = df_prev.groupby('업종 카테고리')['사용 금액'].sum()
this_cat_cnt = df_this.groupby('업종 카테고리').size()

all_cats = sorted(set(this_cat.index) | set(prev_cat.index))
cat_rows = []
for cat in all_cats:
    ta   = float(this_cat.get(cat, 0))
    pa   = float(prev_cat.get(cat, 0))
    diff = ta - pa
    diff_rate_cat = (diff / pa * 100) if pa != 0 else 0.0
    ratio = (ta / this_total * 100) if this_total != 0 else 0.0
    cat_rows.append({
        'category':           cat,
        'total_amount':       int(ta),
        'ratio_percent':      round(ratio, 4),
        'transaction_count':  int(this_cat_cnt.get(cat, 0)),
        'prev_week_amount':   int(pa),
        'diff_amount':        int(diff),
        'diff_rate_percent':  round(diff_rate_cat, 4),
    })

cat_rows.sort(key=lambda x: x['total_amount'], reverse=True)

print(f"{'카테고리':<8} {'이번주':>12} {'전주':>12} {'증감':>12} {'증감율':>8} {'비중':>6}")
print('-' * 62)
for r in cat_rows:
    print(f"{r['category']:<8} {r['total_amount']:>12,.0f} {r['prev_week_amount']:>12,.0f} "
          f"{r['diff_amount']:>+12,.0f} {r['diff_rate_percent']:>+7.1f}% {r['ratio_percent']:>5.1f}%")

top_inc = max((r for r in cat_rows if r['diff_amount'] > 0), key=lambda x: x['diff_amount'], default=None)
top_dec = min((r for r in cat_rows if r['diff_amount'] < 0), key=lambda x: x['diff_amount'], default=None)
print(f"\n  ▲ 가장 많이 증가: {top_inc['category'] if top_inc else '-'} ({top_inc['diff_amount']:+,.0f}원)" if top_inc else "\n  ▲ 증가 카테고리 없음")
print(f"  ▼ 가장 많이 감소: {top_dec['category'] if top_dec else '-'} ({top_dec['diff_amount']:+,.0f}원)" if top_dec else "  ▼ 감소 카테고리 없음")

카테고리              이번주           전주           증감      증감율     비중
--------------------------------------------------------------
식비            330,625      226,478     +104,147   +46.0%  58.9%
생활             96,900            0      +96,900    +0.0%  17.3%
쇼핑             63,352       98,924      -35,572   -36.0%  11.3%
의료             40,830            0      +40,830    +0.0%   7.3%
교통             29,312       26,939       +2,373    +8.8%   5.2%

  ▲ 가장 많이 증가: 식비 (+104,147원)
  ▼ 가장 많이 감소: 쇼핑 (-35,572원)


## [3] 반복 소비 패턴 탐지
TOP 가맹점, N일 연속 방문, 배달/카페/편의점/택시 소비를 탐지합니다.

In [5]:
def is_match(merchant: str, keywords: list[str]) -> bool:
    return any(kw in str(merchant) for kw in keywords)

def merchant_summary(df: pd.DataFrame, keywords: list[str]) -> dict:
    mask = df['결제 내역'].apply(lambda x: is_match(x, keywords))
    sub  = df[mask]
    return {
        'count':               int(len(sub)),
        'total_amount':        int(sub['사용 금액'].sum()),
        'avg_per_transaction': round(float(sub['사용 금액'].mean()), 2) if len(sub) > 0 else 0.0,
    }

def find_consecutive(df: pd.DataFrame, min_streak: int = 2) -> list[dict]:
    """N일 연속 소비된 가맹점 탐지."""
    if df.empty:
        return []
    by_date = df.groupby('결제 내역')['date'].apply(lambda x: sorted(x.unique()))
    result  = []
    for merchant, dates in by_date.items():
        if len(dates) < min_streak:
            continue
        max_s = cur_s = 1
        for i in range(1, len(dates)):
            if (dates[i] - dates[i - 1]).days == 1:
                cur_s += 1
                max_s  = max(max_s, cur_s)
            else:
                cur_s = 1
        if max_s >= min_streak:
            result.append({'merchant': str(merchant), 'max_consecutive_days': max_s})
    return sorted(result, key=lambda x: x['max_consecutive_days'], reverse=True)

# TOP 10 가맹점 (방문 횟수 기준)
top_merch = (
    df_this.groupby('결제 내역')
    .agg(visit_count=('사용 금액', 'count'), total_amount=('사용 금액', 'sum'))
    .reset_index()
    .sort_values('visit_count', ascending=False)
    .head(10)
)
print("[TOP 10 가맹점]")
for _, r in top_merch.iterrows():
    cat_series = df_this[df_this['결제 내역'] == r['결제 내역']]['업종 카테고리'].mode()
    cat_str    = cat_series.iloc[0] if not cat_series.empty else '-'
    print(f"  {r['결제 내역']:15s} {r['visit_count']}회 / {r['total_amount']:>9,.0f}원  [{cat_str}]")

# 연속 방문 가맹점
consecutive = find_consecutive(df_this)
print(f"\n[N일 연속 소비 가맹점 (2일 이상)]")
for c in consecutive:
    print(f"  {c['merchant']}: {c['max_consecutive_days']}일 연속")
if not consecutive:
    print("  (해당 없음)")

# 유형별 소비 요약
delivery = merchant_summary(df_this, DELIVERY_KEYWORDS)
cafe     = merchant_summary(df_this, CAFE_KEYWORDS)
conv     = merchant_summary(df_this, CONVENIENCE_KEYWORDS)
taxi     = merchant_summary(df_this, TAXI_KEYWORDS)

print(f"\n[유형별 소비 요약]")
print(f"  배달음식 : {delivery['count']}회 / {delivery['total_amount']:>10,.0f}원  (건당 평균 {delivery['avg_per_transaction']:,.0f}원)")
print(f"  카페     : {cafe['count']}회 / {cafe['total_amount']:>10,.0f}원  (건당 평균 {cafe['avg_per_transaction']:,.0f}원)")
print(f"  편의점   : {conv['count']}회 / {conv['total_amount']:>10,.0f}원")
print(f"  택시     : {taxi['count']}회 / {taxi['total_amount']:>10,.0f}원")

[TOP 10 가맹점]
  배달의민족           6회 /   138,679원  [식비]
  스타벅스            6회 /    31,989원  [식비]
  맥도날드            4회 /    42,702원  [식비]
  이디야             3회 /    12,504원  [식비]
  서브웨이            3회 /    31,672원  [식비]
  내과              3회 /    40,830원  [의료]
  쿠팡이츠            2회 /    47,256원  [식비]
  CU              2회 /    14,461원  [식비]
  투썸플레이스          2회 /    11,362원  [식비]
  다이소             2회 /    11,450원  [쇼핑]

[N일 연속 소비 가맹점 (2일 이상)]
  스타벅스: 5일 연속
  배달의민족: 4일 연속
  서브웨이: 3일 연속
  다이소: 2일 연속
  맥도날드: 2일 연속

[유형별 소비 요약]
  배달음식 : 8회 /    185,935원  (건당 평균 23,242원)
  카페     : 11회 /     55,855원  (건당 평균 5,078원)
  편의점   : 3회 /     22,081원
  택시     : 2회 /     26,576원


## [4] 요일별 소비 패턴
월~일 요일별 지출을 비교하고, 평일 vs 주말 소비 차이를 분석합니다.

In [6]:
wd_amt = df_this.groupby('weekday')['사용 금액'].sum()
wd_cnt = df_this.groupby('weekday')['사용 금액'].count()

weekday_rows = []
for i, name in enumerate(WEEKDAY_NAMES):
    weekday_rows.append({
        'weekday':           name,
        'weekday_num':       i,
        'total_amount':      int(wd_amt.get(i, 0)),
        'transaction_count': int(wd_cnt.get(i, 0)),
    })

peak_wd_idx  = int(wd_amt.idxmax()) if not wd_amt.empty else None
peak_wd_name = WEEKDAY_NAMES[peak_wd_idx] if peak_wd_idx is not None else None

weekday_total  = sum(float(wd_amt.get(i, 0)) for i in range(5))
weekday_active = sum(1 for i in range(5) if wd_amt.get(i, 0) > 0)
weekend_total  = sum(float(wd_amt.get(i, 0)) for i in range(5, 7))
weekend_active = sum(1 for i in range(5, 7) if wd_amt.get(i, 0) > 0)

weekday_avg = weekday_total / weekday_active if weekday_active > 0 else 0.0
weekend_avg = weekend_total / weekend_active if weekend_active > 0 else 0.0

max_bar = max((r['total_amount'] for r in weekday_rows), default=1)
print("[요일별 소비]")
for r in weekday_rows:
    bar_len = int(r['total_amount'] / max_bar * 30) if max_bar > 0 else 0
    bar     = '█' * bar_len
    print(f"  {r['weekday']}요일 {r['total_amount']:>10,.0f}원  {bar}")
print(f"\n  소비 피크 요일 : {peak_wd_name}요일")
print(f"  평일 평균      : {weekday_avg:,.0f}원")
print(f"  주말 평균      : {weekend_avg:,.0f}원")
print(f"  평일 vs 주말 차이 : {weekend_avg - weekday_avg:+,.0f}원")

[요일별 소비]
  월요일    133,044원  ██████████████████████████████
  화요일     60,588원  █████████████
  수요일     59,946원  █████████████
  목요일     53,026원  ███████████
  금요일     55,464원  ████████████
  토요일     94,436원  █████████████████████
  일요일    104,515원  ███████████████████████

  소비 피크 요일 : 월요일
  평일 평균      : 72,414원
  주말 평균      : 99,476원
  평일 vs 주말 차이 : +27,062원


## [5] 낭비성 소비 탐지
야간 소비, 소액 다빈도, 주간 고액 누적 지출을 탐지합니다.

In [7]:
df_late  = df_this[df_this['hour'] >= LATE_NIGHT_HOUR].copy()
df_micro = df_this[df_this['사용 금액'] < MICRO_THRESHOLD].copy()
df_high  = df_this[df_this['사용 금액'] > upper_bound].copy()

# 야간 소비 상위 카테고리
late_top      = df_late.groupby('업종 카테고리')['사용 금액'].sum().sort_values(ascending=False).head(3)
late_top_list = [{'category': str(c), 'amount': int(a)} for c, a in late_top.items()]

# 소액 다빈도 상위 카테고리
micro_top      = df_micro.groupby('업종 카테고리')['사용 금액'].sum().sort_values(ascending=False).head(3)
micro_top_list = [{'category': str(c), 'amount': int(a)} for c, a in micro_top.items()]

# 주간 고액 지출 누적 (IQR 상한 초과)
high_items = [
    {
        'used_at':  str(r['사용 시간']),
        'merchant': str(r['결제 내역']),
        'amount':   int(r['사용 금액']),
        'category': str(r['업종 카테고리']),
    }
    for _, r in df_high.sort_values('사용 금액', ascending=False).iterrows()
]

print(f"[야간 소비 (21시 이후)]")
print(f"  총액  : {df_late['사용 금액'].sum():,.0f}원 / {len(df_late)}건")
print(f"  비중  : {df_late['사용 금액'].sum() / this_total * 100:.1f}%")
print(f"  주요  : {late_top_list}")

print(f"\n[소액 다빈도 소비 ({MICRO_THRESHOLD:,}원 미만)]")
print(f"  총액  : {df_micro['사용 금액'].sum():,.0f}원 / {len(df_micro)}건")
print(f"  주요  : {micro_top_list}")

print(f"\n[주간 고액 지출 누적 (IQR 상한 {upper_bound:,.0f}원 초과)]")
print(f"  총액  : {df_high['사용 금액'].sum():,.0f}원 / {len(df_high)}건")
for item in high_items:
    print(f"  - {item['used_at']} | {item['merchant']} | {item['amount']:,.0f}원 [{item['category']}]")

[야간 소비 (21시 이후)]
  총액  : 109,121원 / 6건
  비중  : 19.5%
  주요  : [{'category': '식비', 'amount': 91449}, {'category': '교통', 'amount': 17672}]

[소액 다빈도 소비 (10,000원 미만)]
  총액  : 121,089원 / 21건
  주요  : [{'category': '식비', 'amount': 99283}, {'category': '쇼핑', 'amount': 19070}, {'category': '교통', 'amount': 2736}]

[주간 고액 지출 누적 (IQR 상한 22,754원 초과)]
  총액  : 233,443원 / 7건
  - 2024-04-01 10:00:00 | SKT통신비 | 65,000원 [생활]
  - 2024-04-07 19:27:00 | 쿠팡 | 44,282원 [쇼핑]
  - 2024-04-05 18:22:00 | 배달의민족 | 28,701원 [식비]
  - 2024-04-04 21:05:00 | 배달의민족 | 24,432원 [식비]
  - 2024-04-03 20:08:00 | 쿠팡이츠 | 24,370원 [식비]
  - 2024-04-02 22:20:00 | 배달의민족 | 23,772원 [식비]
  - 2024-04-06 21:34:00 | 쿠팡이츠 | 22,886원 [식비]


## [6] 절약 가능성 추정
배달/카페 빈도를 줄였을 때 절약 가능 금액과 전주 대비 개선/악화 카테고리를 출력합니다.

In [8]:
# 배달: 1회 줄이면 절약 예상액
delivery_1less = int(delivery['avg_per_transaction']) if delivery['count'] > 0 else 0

# 카페: 절반으로 줄이면 절약 예상액
cafe_halved = int(cafe['avg_per_transaction'] * (cafe['count'] // 2)) if cafe['count'] > 1 else 0

# 전주 대비 개선/악화
improved = sorted([r for r in cat_rows if r['diff_amount'] < 0], key=lambda x: x['diff_amount'])[:3]
worsened = sorted([r for r in cat_rows if r['diff_amount'] > 0], key=lambda x: x['diff_amount'], reverse=True)[:3]

print("[절약 가능성 추정]")
print(f"  배달 1회 줄이면  : -{delivery_1less:,.0f}원 절약 가능")
print(f"  카페 방문 절반으로: -{cafe_halved:,.0f}원 절약 가능")

print("\n  [전주 대비 개선된 카테고리 ✅]")
for r in improved:
    print(f"    {r['category']}: {abs(r['diff_amount']):,.0f}원 감소")
if not improved:
    print("    (없음)")

print("\n  [전주 대비 악화된 카테고리 ⚠️]")
for r in worsened:
    print(f"    {r['category']}: {r['diff_amount']:,.0f}원 증가")
if not worsened:
    print("    (없음)")

[절약 가능성 추정]
  배달 1회 줄이면  : -23,241원 절약 가능
  카페 방문 절반으로: -25,388원 절약 가능

  [전주 대비 개선된 카테고리 ✅]
    쇼핑: 35,572원 감소

  [전주 대비 악화된 카테고리 ⚠️]
    식비: 104,147원 증가
    생활: 96,900원 증가
    의료: 40,830원 증가


## [7] 주간 지출 마찰력 및 밀도 분석
지출 마찰력(온라인 비중)과 지출 밀도(결제 빈도)를 통해 사용자의 지출 습관을 다각도로 분석합니다.

In [18]:
# 1. 지출 마찰력 (Frictionless Spending): 온라인/간편결제 비중
# 2. 지출 밀도 (Transaction Density): 결제 빈도(건수) vs 금액

# 온라인/간편결제 키워드
FRICTIONLESS_KEYWORDS = ['온라인', '간편결제', '앱결제', '배달']

# 마찰력 분석
is_frictionless = df_this['결제 방식 (온/오프라인)'].str.contains('|'.join(FRICTIONLESS_KEYWORDS), na=False)
df_fric = df_this[is_frictionless]

fric_total = float(df_fric['사용 금액'].sum())
fric_count = len(df_fric)
fric_ratio = (fric_total / this_total * 100) if this_total > 0 else 0.0

# 밀도 분석 (일평균 결제 건수 및 건당 평균 금액)
daily_counts = df_this.groupby('date').size()
avg_daily_count = daily_counts.mean() if not daily_counts.empty else 0
avg_per_swipe = this_total / len(df_this) if len(df_this) > 0 else 0

print("[주간 지출 마찰력 및 밀도 분석]")
print(f"  1. 지출 마찰력 (Pain of Paying)")
print(f"     - 온라인/간편결제: {fric_count}건 / {fric_total:,.0f}원")
print(f"     - 마찰력 없는 지출 비중: {fric_ratio:.1f}%")


print(f"\n  2. 지출 밀도 (Transaction Density)")
print(f"     - 하루 평균 결제 횟수: {avg_daily_count:.1f}회")
print(f"     - 1회 결제당 평균 금액: {avg_per_swipe:,.0f}원")



[주간 지출 마찰력 및 밀도 분석]
  1. 지출 마찰력 (Pain of Paying)
     - 온라인/간편결제: 16건 / 225,108원
     - 마찰력 없는 지출 비중: 40.1%

  2. 지출 밀도 (Transaction Density)
     - 하루 평균 결제 횟수: 6.0회
     - 1회 결제당 평균 금액: 13,358원
